## Load Libraries

In [80]:
import pandas as pd
import numpy as np

## Helper Functions

In [ ]:
column_translation_weather = {
    "ID OMM station": "WMO_Station_ID",
    "Date": "Date",
    "Pression au niveau mer": "Sea_Level_Pressure_hPa",
    "Variation de pression en 3 heures": "Pressure_Change_3h_hPa",
    "Type de tendance barométrique": "Barometric_Trend_Type",
    "Direction du vent moyen 10 mn": "Wind_Direction_10min_deg",
    "Vitesse du vent moyen 10 mn": "Wind_Speed_10min_mps",
    "Température": "Temperature_C",
    "Point de rosée": "Dew_Point_C",
    "Humidité": "Relative_Humidity_percent",
    "Visibilité horizontale": "Horizontal_Visibility_m",
    "Temps présent": "Current_Weather",
    "Temps passé 1": "Past_Weather_1",
    "Temps passé 2": "Past_Weather_2",
    "Nebulosité totale": "Total_Cloud_Cover_okta",
    "Nébulosité des nuages de l' étage inférieur": "Low_Level_Cloud_Cover_okta",
    "Hauteur de la base des nuages de l'étage inférieur": "Low_Cloud_Base_Height_m",
    "Type des nuages de l'étage inférieur": "Low_Cloud_Type",
    "Type des nuages de l'étage moyen": "Mid_Cloud_Type",
    "Type des nuages de l'étage supérieur": "High_Cloud_Type",
    "Pression station": "Station_Pressure_hPa",
    "Niveau barométrique": "Barometric_Level",
    "Géopotentiel": "Geopotential_m2s2",
    "Variation de pression en 24 heures": "Pressure_Change_24h_hPa",
    "Température minimale sur 12 heures": "Min_Temperature_12h_C",
    "Température minimale sur 24 heures": "Min_Temperature_24h_C",
    "Température maximale sur 12 heures": "Max_Temperature_12h_C",
    "Température maximale sur 24 heures": "Max_Temperature_24h_C",
    "Température minimale du sol sur 12 heures": "Min_Ground_Temperature_12h_C",
    "Méthode de mesure Température du thermomètre mouillé": "Wet_Bulb_Measurement_Method",
    "Température du thermomètre mouillé": "Wet_Bulb_Temperature_C",
    "Rafale sur les 10 dernières minutes": "Wind_Gust_Last10min_mps",
    "Rafales sur une période": "Wind_Gust_Period_mps",
    "Periode de mesure de la rafale": "Gust_Measurement_Period",
    "Etat du sol": "Ground_State",
    "Hauteur totale de la couche de neige, glace, autre au sol": "Total_Snow_Ice_Depth_cm",
    "Hauteur de la neige fraîche": "Fresh_Snow_Depth_cm",
    "Periode de mesure de la neige fraiche": "Fresh_Snow_Measurement_Period",
    "Précipitations dans la dernière heure": "Precipitation_Last1h_mm",
    "Précipitations dans les 3 dernières heures": "Precipitation_Last3h_mm",
    "Précipitations dans les 6 dernières heures": "Precipitation_Last6h_mm",
    "Précipitations dans les 12 dernières heures": "Precipitation_Last12h_mm",
    "Précipitations dans les 24 dernières heures": "Precipitation_Last24h_mm",
    "Phénomène spécial 1": "Special_Weather_Event_1",
    "Phénomène spécial 2": "Special_Weather_Event_2",
    "Phénomène spécial 3": "Special_Weather_Event_3",
    "Phénomène spécial 4": "Special_Weather_Event_4",
    "Nébulosité couche nuageuse 1": "Cloud_Layer1_Cover_okta",
    "Type nuage 1": "Cloud_Layer1_Type",
    "Hauteur de base 1": "Cloud_Layer1_Base_Height_m",
    "Nébulosité couche nuageuse 2": "Cloud_Layer2_Cover_okta",
    "Type nuage 2": "Cloud_Layer2_Type",
    "Hauteur de base 2": "Cloud_Layer2_Base_Height_m",
    "Nébulosité couche nuageuse 3": "Cloud_Layer3_Cover_okta",
    "Type nuage 3": "Cloud_Layer3_Type",
    "Hauteur de base 3": "Cloud_Layer3_Base_Height_m",
    "Nébulosité couche nuageuse 4": "Cloud_Layer4_Cover_okta",
    "Type nuage 4": "Cloud_Layer4_Type",
    "Hauteur de base 4": "Cloud_Layer4_Base_Height_m",
    "Nom": "Station_Name",
    "Coordonnees": "Coordinates",
    "Latitude": "Latitude",
    "Longitude": "Longitude",
    "Altitude": "Altitude_m",
    "communes (name)": "Municipality_Name",
    "communes (code)": "Municipality_Code",
    "EPCI (name)": "Intermunicipal_Group_Name",
    "EPCI (code)": "Intermunicipal_Group_Code",
    "department (name)": "Department_Name",
    "department (code)": "Department_Code",
    "region (name)": "Region_Name",
    "region (code)": "Region_Code",
    "mois_de_l_annee": "Month_of_Year"
}


import pandas as pd
import numpy as np

def load_and_clean(file_path):

    # Load correctly
    df = pd.read_csv(file_path, sep=",", encoding="utf-8-sig")

    # Clean column names
    df.columns = df.columns.str.strip()

    # Rename columns
    df.rename(columns=column_translation, inplace=True)

    # Replace ND with NaN
    df.replace("ND", np.nan, inplace=True)

    # Keep only France
    df = df[df["Region"] == "France"]

    # Keep only final validated data
    df = df[df["Data_Type"].str.contains("Donn", na=False)]

    # Create datetime
    df["Datetime"] = pd.to_datetime(
        df["Date"] + " " + df["Time"],
        dayfirst=True,
        errors="coerce"
    )

    # Drop rows without consumption (removes :15 and :45 empty rows)
    df = df[df["Consumption_MW"].notna()]

    # ---- KEEP ONLY VARIABLES USEFUL FOR DEMAND PREDICTION ----
    df = df[[
        "Datetime",
        "Consumption_MW",
        "Forecast_DayMinus1_MW",
        "Forecast_Day_MW"
    ]]

    # Convert numeric
    df["Consumption_MW"] = pd.to_numeric(df["Consumption_MW"], errors="coerce")
    df["Forecast_DayMinus1_MW"] = pd.to_numeric(df["Forecast_DayMinus1_MW"], errors="coerce")
    df["Forecast_Day_MW"] = pd.to_numeric(df["Forecast_Day_MW"], errors="coerce")

    # Sort by time
    df = df.sort_values("Datetime").reset_index(drop=True)

    # TIME FEATURES
    df["Hour"] = df["Datetime"].dt.hour
    df["Minute"] = df["Datetime"].dt.minute
    df["DayOfWeek"] = df["Datetime"].dt.dayofweek
    df["Month"] = df["Datetime"].dt.month
    df["DayOfYear"] = df["Datetime"].dt.dayofyear
    df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)

    # LAG FEATURES (15-min frequency)
    df = df.sort_values("Datetime")
    df["Lag_1"] = df["Consumption_MW"].shift(1)      # 15 min
    df["Lag_4"] = df["Consumption_MW"].shift(4)      # 1 hour
    df["Lag_96"] = df["Consumption_MW"].shift(96)    # 1 day
    df["Lag_672"] = df["Consumption_MW"].shift(672)  # 1 week

    # Drop rows where lags are NaN (first week will be removed)
    df = df.dropna(subset=["Lag_1", "Lag_4", "Lag_96", "Lag_672"]).reset_index(drop=True)

    return df

## Read all csv files and return a combined DataFrame

In [82]:
files = [
    "electricity/eCO2mix_RTE_Annuel-Definitif_2019.csv",
    "electricity/eCO2mix_RTE_Annuel-Definitif_2020.csv",
    "electricity/eCO2mix_RTE_Annuel-Definitif_2021.csv",
    "electricity/eCO2mix_RTE_Annuel-Definitif_2022.csv",
    "electricity/eCO2mix_RTE_Annuel-Definitif_2023.csv"
]

dfs = [load_and_clean(file) for file in files]
df_final = pd.concat(dfs, ignore_index=True)

df_final

# Save to CSV
df_final.to_csv("electricity/France_Electricity_Cleaned.csv", index=False, encoding="utf-8-sig")

/var/folders/w0/qsrp92ys7rb9d622jmmq_pfr0000gn/T/ipykernel_2331/3936007201.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace("ND", np.nan, inplace=True)
/var/folders/w0/qsrp92ys7rb9d622jmmq_pfr0000gn/T/ipykernel_2331/3936007201.py:69: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Datetime"] = pd.to_datetime(
/var/folders/w0/qsrp92ys7rb9d622jmmq_pfr0000gn/T/ipykernel_2331/3936007201.py:60: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, 